In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

feature_df = pd.read_pickle("/content/drive/MyDrive/SmartRail/features_with_clusters.pkl")
X_pca = np.load("/content/drive/MyDrive/SmartRail/X_pca.npy")
kmeans_final = joblib.load("/content/drive/MyDrive/SmartRail/kmeans.pkl")

print(feature_df.shape, X_pca.shape)

Mounted at /content/drive
(1459444, 44) (1459444, 10)


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

sample_idx = np.random.choice(X_pca.shape[0], 2000, replace=False)
X_sample = X_pca[sample_idx]

agglo = AgglomerativeClustering(n_clusters=4)
agglo_labels = agglo.fit_predict(X_sample)

km_labels = kmeans_final.predict(X_sample)
print(f"ARI (KMeans vs Agglomerative): {adjusted_rand_score(km_labels, agglo_labels):.4f}")

ARI (KMeans vs Agglomerative): 0.9858


In [ ]:
from sklearn.cluster import DBSCAN

sample_idx2 = np.random.choice(X_pca.shape[0], 15000, replace=False)
X_sample2 = X_pca[sample_idx2]

dbscan = DBSCAN(eps=1.5, min_samples=10)
db_labels = dbscan.fit_predict(X_sample2)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)

print(f"Clusters found: {n_clusters}")
print(f"Noise/anomaly points: {n_noise} ({n_noise/len(db_labels)*100:.2f}%)")

Clusters found: 13
Noise/anomaly points: 571 (3.81%)


In [ ]:
sample_failures = feature_df["failure"].values[sample_idx2]

anomaly_mask = (db_labels == -1)
print(f"Failure rate in anomaly points: {sample_failures[anomaly_mask].mean()*100:.2f}%")
print(f"Failure rate in normal points: {sample_failures[~anomaly_mask].mean()*100:.2f}%")

Failure rate in anomaly points: 3.85%
Failure rate in normal points: 2.09%


In [ ]:
from sklearn.metrics import pairwise_distances_argmin_min

centers = kmeans_final.cluster_centers_
_, distances = pairwise_distances_argmin_min(X_pca, centers)

feature_df["pca_distance_score"] = distances
feature_df["pca_distance_score"].describe()

,pca_distance_score
count,1.459444e+06
mean,3.412415e+00
std,2.434558e+00
min,4.904128e-01
25%,2.203488e+00
50%,3.016005e+00
75%,4.051104e+00
max,1.825453e+02


In [ ]:
correlation = feature_df["pca_distance_score"].corr(feature_df["failure"])
print(f"Correlation (distance vs failure): {correlation:.4f}")

top5pct_threshold = feature_df["pca_distance_score"].quantile(0.95)
high_distance = feature_df["pca_distance_score"] >= top5pct_threshold

print(f"Failure rate in top 5% distance points: {feature_df.loc[high_distance, 'failure'].mean()*100:.2f}%")
print(f"Failure rate in rest: {feature_df.loc[~high_distance, 'failure'].mean()*100:.2f}%")

Correlation (distance vs failure): -0.0107
Failure rate in top 5% distance points: 2.48%
Failure rate in rest: 2.02%


In [ ]:
feature_df.to_pickle("/content/drive/MyDrive/SmartRail/features_full.pkl")